# 00 — Environment Check

**Project:** CVF25PRJ10  
**Task:** Foundation  
**State:** Template / orchestration notebook. Core logic belongs in `src/`.

Kiểm tra runtime Colab/local, dependency và baseline config trước khi chạy các notebook khác.


## Colab usage rule

Do not duplicate model/data-processing logic in this notebook. Import reusable code from `src/`. Do not commit raw private videos or large checkpoints to Git.


In [ ]:
# Optional when running in Google Colab
# from google.colab import drive
# drive.mount('/content/drive')

from pathlib import Path
import os
import sys

def _is_project_root(path: Path) -> bool:
    return (
        (path / 'AGENTS.md').is_file()
        and (path / 'configs' / 'baseline.yaml').is_file()
        and (path / 'scripts' / 'check_setup.py').is_file()
        and (path / 'src').is_dir()
    )

def find_project_root(start: Path | None = None) -> Path:
    env_root = os.environ.get('CVF_PROJECT_ROOT')
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if _is_project_root(candidate):
            return candidate
        raise RuntimeError(f'CVF_PROJECT_ROOT is not a valid project root: {candidate}')

    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if _is_project_root(candidate):
            return candidate

    # Typical Colab case: cwd=/content and the cloned repo is one direct child.
    child_matches = [
        child for child in start.iterdir()
        if child.is_dir() and _is_project_root(child)
    ] if start.is_dir() else []
    if len(child_matches) == 1:
        return child_matches[0]
    if len(child_matches) > 1:
        raise RuntimeError(
            'Multiple project roots found. Set CVF_PROJECT_ROOT explicitly: ' +
            ', '.join(map(str, child_matches))
        )

    raise RuntimeError(
        'Project root not found. Clone/open the repository, or set CVF_PROJECT_ROOT.'
    )

ROOT = find_project_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('Repository root:', ROOT)


In [ ]:
# Install project dependencies in a fresh Colab runtime when needed.
# !pip install -r "{ROOT / 'requirements.txt'}"


In [ ]:
import platform
print('Python:', platform.python_version())

try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as e:
    print('PyTorch check failed:', e)


In [ ]:
# Repository contract checks
import subprocess
subprocess.run([sys.executable, str(ROOT / 'scripts' / 'check_setup.py')], cwd=ROOT, check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=ROOT, check=True)


In [ ]:
import yaml
from pathlib import Path
config = yaml.safe_load((ROOT / 'configs' / 'baseline.yaml').read_text(encoding='utf-8'))
config
